# 01 Cleaning


Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [ ]:
import pandas as pd
import os

RAW = "../data/raw/"
PROCESSED = "../data/processed/"
os.makedirs(PROCESSED, exist_ok=True) 
orders = pd.read_csv(RAW + "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW + "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW + "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW + "olist_sellers_dataset.csv")
products = pd.read_csv(RAW + "olist_products_dataset.csv")
payments = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
category_tr = pd.read_csv(RAW + "product_category_name_translation.csv")

print("All files loaded successfully")

All files loaded successfully


In [39]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col]=pd.to_datetime(orders[col])

print("Orders date columns fixed")
print(orders.dtypes)

Orders date columns fixed
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [42]:

products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
}, inplace=True)

for col in ['product_weight_g','product_length_cm','product_height_cm','product_width_cm']:
    if col in products.columns:
        products[col]=products[col].fillna(products[col].median())

if 'product_category_name_english' in products.columns:
    products['product_category_name_english']=products['product_category_name_english'].fillna('unknown')

elif 'product_category_name' in products.columns:
    products['product_category_name']=products['product_category_name'].fillna('unknown')

products.drop(columns=[
    'product_name_length',
    'product_description_length',
    'product_photos_qty'
], errors='ignore', inplace=True)

print("Products cleaned")
print(products.head(2))

Products cleaned
                         product_id product_category_name  product_weight_g  \
0  1e9e8ef04dbcff4541ed26657ea517e5            perfumaria             225.0   
1  3aa071139cb16b67ca9e5dea641aaa2f                 artes            1000.0   

   product_length_cm  product_height_cm  product_width_cm  
0               16.0               10.0              14.0  
1               30.0               18.0              20.0  


In [43]:
geolocation = geolocation.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat','mean'),
    geolocation_lng=('geolocation_lng','mean')
).reset_index()
print(f"Geolocation cleaned — {len(geolocation)} unique zip codes")

Geolocation cleaned — 19015 unique zip codes


In [48]:
reviews['review_comment_message']=reviews['review_comment_message'].fillna('no comment')
reviews = reviews[['order_id','review_score','review_comment_message']]

print("Reviews cleaned")

Reviews cleaned


In [49]:
order_items['shipping_limit_date']=pd.to_datetime(order_items['shipping_limit_date'])

print("Order items cleaned")

Order items cleaned


In [51]:
master = orders.copy()
master = master.merge(order_items, on='order_id',how='left')

master = master.merge(products,on='product_id',how='left')

master = master.merge(sellers, on='seller_id',how='left')

master = master.merge(customers,on='customer_id',how='left')

payments_agg =payments.groupby('order_id').agg(
    payment_type=('payment_type','first'),
    payment_installments=('payment_installments','sum'),
    payment_value=('payment_value','sum')
).reset_index()
master= master.merge(payments_agg,on='order_id',how='left')

master =master.merge(reviews,on='order_id',how='left')

master = master.merge(
    geolocation.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'geolocation_lat': 'customer_lat',
        'geolocation_lng': 'customer_lng'
    }),
    on='customer_zip_code_prefix',how='left'
)
print(f"Master dataset created")
print(f"Shape:{master.shape}")
print(f"Columns:{list(master.columns)}")

Master dataset created
Shape:(114092, 33)
Columns:['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_type', 'payment_installments', 'payment_value', 'review_score', 'review_comment_message', 'customer_lat', 'customer_lng']


In [52]:
print("Null counts in master:")
print(master.isnull().sum()[master.isnull().sum()>0])
master.to_csv(PROCESSED + "cleaned_master.csv", index=False)
print("\n cleaned_master.csv saved to data/processed/")

Null counts in master:
order_approved_at                 162
order_delivered_carrier_date     1980
order_delivered_customer_date    3253
order_item_id                     778
product_id                        778
seller_id                         778
shipping_limit_date               778
price                             778
freight_value                     778
product_category_name             778
product_weight_g                  778
product_length_cm                 778
product_height_cm                 778
product_width_cm                  778
seller_zip_code_prefix            778
seller_city                       778
seller_state                      778
payment_type                        3
payment_installments                3
payment_value                       3
review_score                      961
review_comment_message            961
customer_lat                      311
customer_lng                      311
dtype: int64

 cleaned_master.csv saved to data/processed/
